# Querycraft v2 notebook C: 4 bit AWQ (W4A16) of the merged 7B DoRA model

Settings: **GPU T4 x2** (or Colab T4 as a backup), **Internet on**. Same three inputs. Quantizes with llm-compressor's
AWQ recipe, calibrated on 64 val prompts (up to 1,024 tokens) with their gold SQL, as for the 1.5B model (val is the split choices are made on; dev and held
back are never touched). Output: `/kaggle/working/out/qwen7b_dora_awq.tar.gz` (~5.5 GB), which the laptop benchmark
loads with vLLM (`make bench`). Estimated 1 GPU hour.


In [ ]:
import glob, json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path
def sh(cmd):
    print("$", cmd, flush=True); subprocess.run(cmd, shell=True, check=True)
sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv")
IN = Path("/kaggle/input")
def find(slug):
    hits = [p for p in IN.glob("*") if slug in p.name] + [p for p in IN.glob("*/*/*") if slug in str(p)]
    assert hits, f"attach the dataset {slug} (Add Input)"; return hits[0]
W, T = Path("/kaggle/working"), Path("/kaggle/tmp")
QC = T / "qc"; OUT = W / "out"  # code + data in tmp (not saved); only out/ is kept
OUT.mkdir(exist_ok=True); T.mkdir(exist_ok=True)
def unpack(src, dst):
    """Kaggle may keep uploaded folders as .zip files: extract those, copy the rest."""
    dst.mkdir(parents=True, exist_ok=True)
    for p in Path(src).iterdir():
        if p.suffix == ".zip":
            zipfile.ZipFile(p).extractall(dst)
        elif p.is_dir():
            shutil.copytree(p, dst / p.name, dirs_exist_ok=True)
        else:
            shutil.copy(p, dst / p.name)
unpack(find("querycraft-code"), QC)
unpack(find("querycraft-bird-eval"), QC)            # data/raw/... and data/processed/...
unpack(find("querycraft-7b-dora"), T / "runs/qwen7b_dora_best/ckpt")
os.chdir(QC); os.environ["PYTHONPATH"] = str(QC); sys.path.insert(0, str(QC))
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
(QC / "runs").mkdir(exist_ok=True)
print(sorted(p.name for p in QC.iterdir()))


In [ ]:
sh("pip install -q uv && uv pip install --system -q llmcompressor 'transformers>=5.16.1,<5.20.0' peft sqlglot pyyaml")


In [ ]:
# DoRA is not servable by vLLM: merge it into the 7B base on CPU (bf16 weights, ~15 GB in /kaggle/tmp)
from querycraft.infer import merge_adapter
t0 = time.time()
MERGED = merge_adapter("Qwen/Qwen2.5-Coder-7B-Instruct", str(T / "runs/qwen7b_dora_best/ckpt/checkpoint-560"))
print(MERGED, f"{time.time() - t0:.0f} s")


In [ ]:
import torch
from datasets import Dataset
from llmcompressor import oneshot
from llmcompressor.modifiers.awq import AWQModifier
from transformers import AutoModelForCausalLM, AutoTokenizer
from querycraft.data import load_split
from querycraft.prompts import PromptBuilder
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Coder-7B-Instruct")
pb = PromptBuilder(tok, 1280)
rows = load_split("val")[:64]
texts = [tok.apply_chat_template(pb.messages(r) + [{"role": "assistant", "content": r["sql"]}], tokenize=False) for r in rows]
ds = Dataset.from_dict({"text": texts}).map(lambda x: tok(x["text"], truncation=True, max_length=1024), remove_columns=["text"])
model = AutoModelForCausalLM.from_pretrained(MERGED, dtype=torch.float16, device_map="auto")
recipe = [AWQModifier(targets=["Linear"], scheme="W4A16_ASYM", ignore=["lm_head"])]
t0 = time.time()
oneshot(model=model, dataset=ds, recipe=recipe, max_seq_length=1024, num_calibration_samples=len(texts))
dst = W / "qwen7b_dora_awq"
model.save_pretrained(dst, save_compressed=True); tok.save_pretrained(dst)
json.dump({"calibration": "64 val prompts + gold SQL, 1024 tokens", "scheme": "W4A16_ASYM", "seconds": round(time.time() - t0)},
          open(dst / "querycraft_quant.json", "w"))
sh(f"cd {W} && tar czf {OUT}/qwen7b_dora_awq.tar.gz qwen7b_dora_awq && rm -rf qwen7b_dora_awq")
